# Chương 3: Mạng Nơ-ron Tích Chập (CNN) & Thị Giác Máy Tính
**Học viện Công nghệ Bưu chính Viễn thông (PTIT)**
*Môn học: Phát triển các Hệ thống Thông minh (GVHD: PGS.TS Trần Đình Quế)*
*Sinh viên: Phạm Văn Tư - B23DCCN879*

## 1. Toán Học Phép Tích Chập 2D Scratch Bằng NumPy
Hiện thực hóa hàm tích chập 2D với bước trượt (Stride) và đệm (Padding):

In [1]:
import numpy as np
import matplotlib.pyplot as plt

def conv2d_scratch(img, kernel, stride=1, padding=0):
    if padding > 0:
        img_padded = np.pad(img, ((padding, padding), (padding, padding)), mode='constant')
    else:
        img_padded = img
    H, W = img_padded.shape
    kH, kW = kernel.shape
    outH = (H - kH) // stride + 1
    outW = (W - kW) // stride + 1
    output = np.zeros((outH, outW))
    for i in range(outH):
        for j in range(outW):
            region = img_padded[i*stride:i*stride+kH, j*stride:j*stride+kW]
            output[i, j] = np.sum(region * kernel)
    return output

toy_img = np.random.rand(8, 8)
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
filtered = conv2d_scratch(toy_img, sobel_x, padding=1)
print("Kích thước đầu ra tích chập:", filtered.shape)


Kích thước đầu ra tích chập: (8, 8)


## 2. Huấn Luyện Mạng ResNet-18 Bằng PyTorch
Xây dựng kiến trúc ResNet-18 với Skip Connections triệt tiêu hiện tượng triệt tiêu đạo hàm:

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        return F.relu(out)

model = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1),
    nn.BatchNorm2d(32),
    nn.ReLU(),
    ResidualBlock(32),
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Linear(32, 10)
)
print(model)


Sequential(
  (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): ResidualBlock(
    (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (4): AdaptiveAvgPool2d(output_size=(1, 1))
  (5): Flatten(start_dim=1, end_dim=-1)
  (6): Linear(in_features=32, out_features=10, bias=True)
)
